S30 - EVIDENCIA DE APRENDIZAJE 1.
ANÁLISIS Y HERRAMIENTAS DE EXTRACCIÓN DE DATOS



DIEGO MAURICIO SUAZA HOLGUÍN



PROGRAMACIÓN PARA ANÁLISIS DE DATOS
(ANDRÉS FELIPE PALACIO) - PREICA2602B020048


INGENIERÍA DE DATOS


UNIVERSIDAD DIGITAL DE ANTIOQUIA


27 de octubre de 2026
Medellín, Antioquia


Técnica 1: BeautifulSoup

In [10]:
!pip -q install requests beautifulsoup4 pandas

In [11]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin

url = "https://books.toscrape.com/"
respuesta = requests.get(url, timeout=30)
respuesta.raise_for_status()

soup = BeautifulSoup(respuesta.text, "html.parser")

productos = []

for libro in soup.select("article.product_pod"):
    nombre = libro.h3.a.get("title", "").strip()
    precio = libro.select_one(".price_color").get_text(strip=True)
    disponibilidad = libro.select_one(".availability").get_text(" ", strip=True)
    enlace = urljoin(url, libro.h3.a.get("href"))

    productos.append({
        "nombre": nombre,
        "precio": precio,
        "disponibilidad": disponibilidad,
        "enlace": enlace
    })

df = pd.DataFrame(productos)
display(df.head(10))

df.to_csv("/content/productos_beautifulsoup.csv",
          index=False, encoding="utf-8-sig")

print("Productos extraídos:", len(df))

DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): books.toscrape.com:443
DEBUG:urllib3.connectionpool:https://books.toscrape.com:443 "GET / HTTP/1.1" 200 None
CRITICAL:twisted:Unhandled error in Deferred:
2026-10-27 12:39:13 [twisted] CRITICAL: Unhandled error in Deferred:
CRITICAL:twisted:Unhandled error in Deferred:
2026-10-27 12:39:13 [twisted] CRITICAL: Unhandled error in Deferred:
CRITICAL:twisted:
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/twisted/internet/defer.py", line 1834, in _inlineCallbacks
    result = context.run(
        cast(Failure, result).throwExceptionIntoGenerator, gen
    )
  File "/usr/local/lib/python3.13/dist-packages/twisted/python/failure.py", line 467, in throwExceptionIntoGenerator
    return g.throw(self.value.with_traceback(self.tb))
           ~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/scrapy/crawler.py", line 564, in _crawl
    yield d
  File "/usr/lo

,nombre,precio,disponibilidad,enlace
0,A Light in the Attic,Â£51.77,In stock,https://books.toscrape.com/catalogue/a-light-i...
1,Tipping the Velvet,Â£53.74,In stock,https://books.toscrape.com/catalogue/tipping-t...
2,Soumission,Â£50.10,In stock,https://books.toscrape.com/catalogue/soumissio...
3,Sharp Objects,Â£47.82,In stock,https://books.toscrape.com/catalogue/sharp-obj...
4,Sapiens: A Brief History of Humankind,Â£54.23,In stock,https://books.toscrape.com/catalogue/sapiens-a...
5,The Requiem Red,Â£22.65,In stock,https://books.toscrape.com/catalogue/the-requi...
6,The Dirty Little Secrets of Getting Your Dream...,Â£33.34,In stock,https://books.toscrape.com/catalogue/the-dirty...
7,The Coming Woman: A Novel Based on the Life of...,Â£17.93,In stock,https://books.toscrape.com/catalogue/the-comin...
8,The Boys in the Boat: Nine Americans and Their...,Â£22.60,In stock,https://books.toscrape.com/catalogue/the-boys-...
9,The Black Maria,Â£52.15,In stock,https://books.toscrape.com/catalogue/the-black...


Productos extraídos: 20


Técnica 2: Selenium
Esta técnica abre la página en un navegador automatizado y extrae los datos visibles.

In [12]:
!pip -q install selenium pandas
!apt-get -qq update
!apt-get -qq install chromium chromium-driver

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


In [13]:
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options

opciones = Options()
opciones.add_argument("--headless")
opciones.add_argument("--no-sandbox")
opciones.add_argument("--disable-dev-shm-usage")

driver = webdriver.Chrome(options=opciones)

try:
    driver.get("https://books.toscrape.com/")

    productos = []

    for libro in driver.find_elements(By.CSS_SELECTOR, "article.product_pod"):
        nombre = libro.find_element(By.CSS_SELECTOR, "h3 a").get_attribute("title")
        precio = libro.find_element(By.CSS_SELECTOR, ".price_color").text
        disponibilidad = libro.find_element(By.CSS_SELECTOR, ".availability").text
        enlace = libro.find_element(By.CSS_SELECTOR, "h3 a").get_attribute("href")

        productos.append({
            "nombre": nombre,
            "precio": precio,
            "disponibilidad": disponibilidad.strip(),
            "enlace": enlace
        })

    df_selenium = pd.DataFrame(productos)
    display(df_selenium.head(10))

    df_selenium.to_csv(
        "/content/productos_selenium.csv",
        index=False,
        encoding="utf-8-sig"
    )

    print("Productos extraídos:", len(df_selenium))

finally:
    driver.quit()

DEBUG:selenium.webdriver.common.selenium_manager:Selenium Manager binary found at: /usr/local/lib/python3.13/dist-packages/selenium/webdriver/common/linux-x86_64/selenium-manager
DEBUG:selenium.webdriver.common.selenium_manager:Executing process: /usr/local/lib/python3.13/dist-packages/selenium/webdriver/common/linux-x86_64/selenium-manager --browser chrome --language-binding python --output json
DEBUG:selenium.webdriver.common.selenium_manager:Driver path: /root/.cache/selenium/chromedriver/linux64/156.0.8078.47/chromedriver
DEBUG:selenium.webdriver.common.selenium_manager:Browser path: /root/.cache/selenium/chrome/linux64/156.0.8078.47/chrome
DEBUG:selenium.webdriver.common.service:Started executable: `/root/.cache/selenium/chromedriver/linux64/156.0.8078.47/chromedriver` in a child process with pid: 11149 using 0 to output -3
DEBUG:selenium.webdriver.remote.remote_connection:POST http://localhost:42599/session {'capabilities': {'firstMatch': [{}], 'alwaysMatch': {'browserName': 'chr

SessionNotCreatedException: Message: session not created: Chrome instance exited. Examine ChromeDriver verbose log to determine the cause.; For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#sessionnotcreatedexception
Stacktrace:
#0 0x5d12ad2287ea <unknown>
#1 0x5d12acb679e5 <unknown>
#2 0x5d12acb98156 <unknown>
#3 0x5d12acb96014 <unknown>
#4 0x5d12acbd6221 <unknown>
#5 0x5d12acbd5ccc <unknown>
#6 0x5d12acba0cd7 <unknown>
#7 0x5d12acb9ffee <unknown>
#8 0x5d12acba0ac1 <unknown>
#9 0x5d12ad1eb7ac <unknown>
#10 0x5d12ad1e9fb5 <unknown>
#11 0x5d12ad1d5315 <unknown>
#12 0x5d12ad1eac1a <unknown>
#13 0x5d12ad1bdbf9 <unknown>
#14 0x5d12ad212ef8 <unknown>
#15 0x5d12ad213095 <unknown>
#16 0x5d12ad227183 <unknown>
#17 0x7aabc680bb84 <unknown>
#18 0x7aabc6898ecc <unknown>


Técnica 3: Scrapy
Scrapy permite recorrer varias páginas del catálogo y extraer más productos automáticamente.

In [14]:
!pip -q install scrapy

In [18]:
import os
import pandas as pd
import scrapy
# Install crochet if not present to run Scrapy inside Jupyter safely
!pip install -q crochet
from scrapy.crawler import CrawlerRunner
from crochet import setup, wait_for

# Initialize crochet background thread loop
setup()

ARCHIVO = "/content/productos_scrapy.csv"

class LibrosSpider(scrapy.Spider):
    name = "libros"
    start_urls = ["https://books.toscrape.com/"]

    custom_settings = {
        "ROBOTSTXT_OBEY": True,
        "DOWNLOAD_DELAY": 0.5,
        "LOG_LEVEL": "INFO",
        # Especificamos explícitamente el reactor compatible con crochet en los settings
        "TWISTED_REACTOR": "twisted.internet.epollreactor.EPollReactor",
        "FEEDS": {
            ARCHIVO: {
                "format": "csv",
                "encoding": "utf-8-sig",
                "overwrite": True
            }
        }
    }

    def parse(self, response):
        print("Página consultada:", response.url)
        print("Código HTTP:", response.status)

        for libro in response.css("article.product_pod"):
            yield {
                "nombre": libro.css("h3 a::attr(title)").get(default=""),
                "precio": libro.css(".price_color::text").get(default=""),
                "disponibilidad": " ".join(
                    texto.strip()
                    for texto in libro.css(".availability::text").getall()
                    if texto.strip()
                ),
                "enlace": response.urljoin(
                    libro.css("h3 a::attr(href)").get(default="")
                )
            }

        siguiente = response.css("li.next a::attr(href)").get()
        if siguiente:
            yield response.follow(siguiente, callback=self.parse)

# Decorator tells crochet to block synchronously up to 45 seconds while crawling in background thread
@wait_for(timeout=45.0)
def ejecutar_scrapy():
    if os.path.exists(ARCHIVO):
        os.remove(ARCHIVO)
    # Indicamos al CrawlerRunner el reactor exacto que ya está activo en Colab
    runner = CrawlerRunner(settings={
        'TWISTED_REACTOR': 'twisted.internet.epollreactor.EPollReactor'
    })
    return runner.crawl(LibrosSpider)

print("Iniciando el rastreo con Scrapy...")
ejecutar_scrapy()

# Comprobar que el archivo exista antes de leerlo
if os.path.exists(ARCHIVO):
    df_scrapy = pd.read_csv(ARCHIVO)
    print("\nArchivo CSV generado correctamente.")
    print("Total de productos extraídos:", len(df_scrapy))
    display(df_scrapy.head(10))
else:
    print("\nNo se generó el archivo CSV.")

INFO:scrapy.addons:Enabled addons:
[]
2026-10-27 12:47:36 [scrapy.addons] INFO: Enabled addons:
[]
DEBUG:scrapy.utils.log:Using reactor: twisted.internet.epollreactor.EPollReactor
INFO:scrapy.extensions.telnet:Telnet Password: db523e4d974086e0
2026-10-27 12:47:36 [scrapy.extensions.telnet] INFO: Telnet Password: db523e4d974086e0
2026-10-27 12:47:36 [scrapy.middleware] WARNING: Disabled scrapy.extensions.remote_control.RemoteControl: RemoteControl requires the asyncio support. You can set the REMOTE_CONTROL_ENABLED setting to False to remove this warning.
INFO:scrapy.middleware:Enabled extensions:
['scrapy.extensions.corestats.CoreStats',
 'scrapy.extensions.logcount.LogCount',
 'scrapy.extensions.telnet.TelnetConsole',
 'scrapy.extensions.memusage.MemoryUsage',
 'scrapy.extensions.feedexport.FeedExporter',
 'scrapy.extensions.logstats.LogStats']
2026-10-27 12:47:36 [scrapy.middleware] INFO: Enabled extensions:
['scrapy.extensions.corestats.CoreStats',
 'scrapy.extensions.logcount.LogCo

Iniciando el rastreo con Scrapy...


Level 5:charset_normalizer:override steps (5) and chunk_size (512) as content does not fit (105 byte(s) given) parameters.
Level 5:charset_normalizer:Code page ascii does not fit given bytes sequence at ALL. 'ascii' codec can't decode byte 0xe6 in position 5: ordinal not in range(128)
Level 5:charset_normalizer:Code page utf_8 does not fit given bytes sequence at ALL. 'utf-8' codec can't decode bytes in position 5-6: invalid continuation byte
Level 5:charset_normalizer:Code page big5 does not fit given bytes sequence at ALL. 'big5' codec can't decode byte 0xfa in position 7: illegal multibyte sequence
Level 5:charset_normalizer:Code page big5hkscs does not fit given bytes sequence at ALL. 'big5hkscs' codec can't decode byte 0xa1 in position 9: illegal multibyte sequence
Level 5:charset_normalizer:Code page cp932 does not fit given bytes sequence at ALL. 'cp932' codec can't decode byte 0x98 in position 21: illegal multibyte sequence
Level 5:charset_normalizer:Code page cp949 does not fi

Página consultada: https://books.toscrape.com/
Código HTTP: 200


Level 5:charset_normalizer:Code page ascii does not fit given bytes sequence at ALL. 'ascii' codec can't decode byte 0xf8 in position 1: ordinal not in range(128)
Level 5:charset_normalizer:Code page utf_8 does not fit given bytes sequence at ALL. 'utf-8' codec can't decode byte 0xf8 in position 1: invalid start byte
Level 5:charset_normalizer:Code page big5 does not fit given bytes sequence at ALL. 'big5' codec can't decode byte 0xf8 in position 1: illegal multibyte sequence
Level 5:charset_normalizer:Code page big5hkscs does not fit given bytes sequence at ALL. 'big5hkscs' codec can't decode byte 0xf8 in position 1: illegal multibyte sequence
Level 5:charset_normalizer:Code page cp932 does not fit given bytes sequence at ALL. 'cp932' codec can't decode byte 0xf8 in position 1: illegal multibyte sequence
Level 5:charset_normalizer:Code page cp949 does not fit given bytes sequence at ALL. 'cp949' codec can't decode byte 0xf8 in position 1: illegal multibyte sequence
Level 5:charset_nor

Página consultada: https://books.toscrape.com/catalogue/page-2.html
Código HTTP: 200


... (93 outputs left out)

Página consultada: https://books.toscrape.com/catalogue/page-49.html
Código HTTP: 200


Level 5:charset_normalizer:Code page ascii does not fit given bytes sequence at ALL. 'ascii' codec can't decode byte 0xf8 in position 1: ordinal not in range(128)
Level 5:charset_normalizer:Code page utf_8 does not fit given bytes sequence at ALL. 'utf-8' codec can't decode byte 0xf8 in position 1: invalid start byte
Level 5:charset_normalizer:Code page big5 does not fit given bytes sequence at ALL. 'big5' codec can't decode byte 0xf8 in position 1: illegal multibyte sequence
Level 5:charset_normalizer:Code page big5hkscs does not fit given bytes sequence at ALL. 'big5hkscs' codec can't decode byte 0xf8 in position 1: illegal multibyte sequence
Level 5:charset_normalizer:Code page cp932 does not fit given bytes sequence at ALL. 'cp932' codec can't decode byte 0xf8 in position 1: illegal multibyte sequence
Level 5:charset_normalizer:Code page cp949 does not fit given bytes sequence at ALL. 'cp949' codec can't decode byte 0xf8 in position 1: illegal multibyte sequence
Level 5:charset_nor

Página consultada: https://books.toscrape.com/catalogue/page-50.html
Código HTTP: 200

Archivo CSV generado correctamente.
Total de productos extraídos: 1000


,nombre,precio,disponibilidad,enlace
0,A Light in the Attic,£51.77,In stock,https://books.toscrape.com/catalogue/a-light-i...
1,Tipping the Velvet,£53.74,In stock,https://books.toscrape.com/catalogue/tipping-t...
2,Soumission,£50.10,In stock,https://books.toscrape.com/catalogue/soumissio...
3,Sharp Objects,£47.82,In stock,https://books.toscrape.com/catalogue/sharp-obj...
4,Sapiens: A Brief History of Humankind,£54.23,In stock,https://books.toscrape.com/catalogue/sapiens-a...
5,The Requiem Red,£22.65,In stock,https://books.toscrape.com/catalogue/the-requi...
6,The Dirty Little Secrets of Getting Your Dream...,£33.34,In stock,https://books.toscrape.com/catalogue/the-dirty...
7,The Coming Woman: A Novel Based on the Life of...,£17.93,In stock,https://books.toscrape.com/catalogue/the-comin...
8,The Boys in the Boat: Nine Americans and Their...,£22.60,In stock,https://books.toscrape.com/catalogue/the-boys-...
9,The Black Maria,£52.15,In stock,https://books.toscrape.com/catalogue/the-black...


In [19]:
from google.colab import files

files.download("/content/productos_scrapy.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>